# NVS Benchmark no Google Colab

[Abrir no Colab](https://colab.research.google.com/github/PedroHeinrichSP/TCC-Source-Code/blob/update/notebooks/nvs_benchmark_colab.ipynb) — clique para abrir e rodar o notebook no Google Colab.

Notebook focado em execucao no Colab: clone, setup, benchmark rapido, relatorio HTML, download e backup opcional no Drive.

## 📋 Como usar

1. **Execute as céulas na ordem** (Colab executa linearmente de cima para baixo)
2. **Célula 1 (esta)**: Documentação — sem ação necessária
3. **Célula 2**: Configure repositório e opções globais (REPO_URL, BRANCH, etc). Geralmente não muda.
4. **Célula 3**: Clone do repositório — não edite
5. **Célula 4**: Setup do ambiente — não edite
6. **Célula 5**: Verificação de GPU — não edite
7. **Célula 6**: Montagem opcional do Google Drive — edite `USE_GOOGLE_DRIVE` aqui se quiser backup
8. **⭐ Célula 7 (NOVA)**: Descoberta e seleção visual — **EDITE AQUI para escolher datasets, métodos, preset e modo de execução**. Caixas de seleção aparecerão se houver widgets disponíveis.
9. **Célula 8**: Download de datasets — executado automaticamente
10. **Célula 9**: Benchmark — usa as seleções da Célula 7. Não edite, apenas execute.
11. **Célula 10**: Relatório HTML — não edite
12. **Célula 11**: Exibição do relatório — não edite
13. **Célula 12**: Download de artefatos — não edite
14. **Célula 13**: Backup no Drive — não edite

## 🎯 O que editar e onde

| O que mudar | Onde | Como |
| --- | --- | --- |
| **URL do repositório ou branch** | Célula 2 | Mude `REPO_URL` ou `BRANCH` |
| **Método (nerf_static, gs_static, etc)** | Célula 7 | Selecione no widget ou edite `SELECTED_METHOD` |
| **Dataset (blender_synthetic, d_nerf, etc)** | Célula 7 | Selecione no widget ou edite `SELECTED_DATASET` |
| **Preset (smoke, quick, preview, standard, full)** | Célula 7 | Selecione no widget ou edite `SELECTED_PRESET` |
| **Modo de execução (quick_check ou full)** | Célula 7 | Selecione no widget ou edite `RUN_MODE` |
| **Modo estrito (resultados validados)** | Célula 7 | Marque/desmarque no widget ou edite `STRICT_RESULTS` |
| **Backup no Google Drive** | Célula 6 | Mude `USE_GOOGLE_DRIVE = True` ou `False` |
| **Gerar PDF do relatório** | Célula 7 | Marque/desmarque no widget ou edite `GENERATE_PDF` |

## ✅ Fluxo de execução

```
[Clone] → [Setup] → [GPU?] → [Drive?] → [Selecione] → [Download datasets] → [Benchmark] → [Relatório] → [Download]
```

**Tempo estimado**: 10–30 min (depende do preset e dataset escolhido).


In [ ]:
# Parametros globais de repositorio e caminhos (geralmente não edite)
REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = "/content/TCC"
BRANCH = "update"
RUN_ID = "colab_quick"
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NVS_Benchmark"

# NOTA: Método, dataset, preset e modo são configurados na Célula 7 (Descoberta e Seleção Visual)
# Não edite aqui. Use a Célula 7 para mudar essas opções.

In [ ]:
# Clone do repositório
# Configurações estão na Célula 2: REPO_URL, BRANCH, REPO_DIR
print("=" * 70)
print(f"Clonando repositório: {REPO_URL} (branch: {BRANCH})")
print("=" * 70)

import os
import shutil
import subprocess
import sys

try:
    __import__("google.colab")
except Exception as exc:
    raise RuntimeError("Este notebook foi desenhado para Google Colab.") from exc

if os.path.exists(REPO_DIR):
    print(f"Removendo pasta existente: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print("Clonando...")
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print(f"\n✓ Projeto clonado em: {os.getcwd()}")

In [ ]:
# Setup do ambiente
print("=" * 70)
print("Instalando dependências...")
print("=" * 70)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

print("\nVerificando instalação...")
subprocess.run([sys.executable, "-m", "nvs_benchmark.cli", "status"], check=True)
print("\n✓ Ambiente pronto.")

In [ ]:
# Montagem opcional do Google Drive
# Mude USE_GOOGLE_DRIVE para True se quiser fazer backup dos artefatos no Drive
# (relatório e métricas serão copiados para /content/drive/MyDrive/NVS_Benchmark)
print("=" * 70)
print("Configuração: Google Drive")
print("=" * 70)
print(f"Backup no Drive: {'SIM' if USE_GOOGLE_DRIVE else 'NÃO'}")

if USE_GOOGLE_DRIVE:
    drive_mod = __import__("google.colab", fromlist=["drive"])
    drive = getattr(drive_mod, "drive")
    drive.mount("/content/drive")
    print("✓ Drive montado com sucesso.")
else:
    print("  (Para ativar, mude USE_GOOGLE_DRIVE = True na Célula 2)")
print()

In [ ]:
# Verificação de GPU no Colab
print("=" * 70)
print("Verificação de Hardware")
print("=" * 70)

import torch
cuda_available = torch.cuda.is_available()
print(f"CUDA disponível: {'SIM ✓' if cuda_available else 'NÃO ✗'}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ GPU não detectada. O benchmark rodará em CPU (mais lento).")
print()

In [ ]:
# 🎯 Descoberta e Seleção Visual de Datasets e Métodos
# Edite aqui para escolher qual método, dataset, preset e modo executar

from pathlib import Path
import json

# ============================================================================
# DEFAULTS (edite estes se os widgets não funcionarem ou para mudar valores)
# ============================================================================
SELECTED_METHOD = "nerf_static"      # Ex: "nerf_static", "gs_static", "gs_dynamic", "nerf_dynamic"
SELECTED_DATASET = "blender_synthetic"  # Ex: "blender_synthetic", "d_nerf", "mipnerf360", "tanks_and_temples"
SELECTED_PRESET = "quick"            # Ex: "smoke", "quick", "preview", "standard", "full"
RUN_MODE = "full"                    # "full" ou "quick_check" para teste rápido
APPLY_COMPATIBILITY_FILTER = True    # Filtrar combos incompatíveis (static methods com datasets dinâmicos)
STRICT_RESULTS = True                # Validar resultados antes de incluir no relatório
GENERATE_PDF = False                 # Gerar PDF além de HTML
MIN_REQUIRED_PAIRS = 1               # Número mínimo de pares de camera-imagem requerido

# ============================================================================
# DESCOBERTA DE DATASETS E MÉTODOS
# ============================================================================

def discover_available_datasets() -> dict[str, str]:
    """Descobrir datasets disponíveis em ./data com seus caminhos raiz."""
    available = {}
    data_dir = Path("./data")
    
    # Padrões de reconhecimento por dataset
    markers = {
        "blender_synthetic": ["transforms_train.json"],
        "d_nerf": ["transforms_train.json"],
        "mipnerf360": ["poses_bounds.npy"],
        "tanks_and_temples": ["transforms_train.json", "poses_bounds.npy"],
        "custom": ["transforms_train.json", "poses_bounds.npy"],
    }
    
    if not data_dir.exists():
        print("[info] ./data não existe ainda. Datasets serão instalados na próxima célula.")
        return available
    
    for dataset_name, marker_list in markers.items():
        for marker in marker_list:
            for found_path in data_dir.rglob(marker):
                parent = found_path.parent
                # Para mipnerf360, validar que está na pasta certa
                if dataset_name == "mipnerf360":
                    path_str = str(parent).lower()
                    if "mipnerf360" not in path_str and "360_v2" not in path_str:
                        continue
                available[dataset_name] = str(parent)
                break
            if dataset_name in available:
                break
    
    return available

def discover_available_methods() -> list[str]:
    """Descobrir métodos disponíveis no registry."""
    try:
        from nvs_benchmark.methods import build_registry_with_all_methods
        registry = build_registry_with_all_methods()
        if hasattr(registry, "list_ids"):
            return registry.list_ids()
        elif hasattr(registry, "keys"):
            return list(registry.keys())
    except Exception as e:
        print(f"[warn] Falha ao descobrir métodos: {e}")
    
    # Fallback: métodos padrão conhecidos
    return ["nerf_static", "nerf_dynamic", "gs_static", "gs_dynamic"]

# Descobrir disponibilidades
available_datasets = discover_available_datasets()
available_methods = discover_available_methods()

print("=" * 70)
print("DESCOBERTA: Datasets e Métodos Disponíveis")
print("=" * 70)
print(f"\nDatasets disponíveis em ./data: {list(available_datasets.keys()) if available_datasets else '[nenhum]'}")
print(f"Métodos disponíveis: {available_methods}")
print()

# ============================================================================
# INTERFACE COM WIDGETS (Colab / IPython)
# ============================================================================

USE_WIDGETS = False
try:
    from ipywidgets import Checkbox, Dropdown, HBox, VBox, Label, Button, Output
    from IPython.display import display, HTML
    USE_WIDGETS = True
except ImportError:
    print("[info] ipywidgets não disponível. Use valores defaults abaixo ou edite as variáveis acima.")

if USE_WIDGETS and (available_datasets or available_methods):
    print("\n" + "=" * 70)
    print("SELEÇÃO VISUAL (clique nas opções abaixo)")
    print("=" * 70 + "\n")
    
    # Widget: Seleção de método
    method_dropdown = Dropdown(
        options=available_methods,
        value=SELECTED_METHOD if SELECTED_METHOD in available_methods else available_methods[0],
        description="Método:",
        disabled=False,
    )
    
    # Widget: Seleção de dataset
    dataset_dropdown = Dropdown(
        options=list(available_datasets.keys()) if available_datasets else ["blender_synthetic"],
        value=SELECTED_DATASET if SELECTED_DATASET in available_datasets else (list(available_datasets.keys())[0] if available_datasets else "blender_synthetic"),
        description="Dataset:",
        disabled=not bool(available_datasets),
    )
    
    # Widget: Seleção de preset
    preset_dropdown = Dropdown(
        options=["smoke", "quick", "preview", "standard", "full"],
        value=SELECTED_PRESET,
        description="Preset:",
        disabled=False,
    )
    
    # Widget: Modo quick_check
    quick_check_checkbox = Checkbox(
        value=(RUN_MODE == "quick_check"),
        description="Teste rápido (quick_check)?",
        indent=False,
    )
    
    # Widget: Strict results
    strict_checkbox = Checkbox(
        value=STRICT_RESULTS,
        description="Modo estrito (validar resultados)?",
        indent=False,
    )
    
    # Widget: Gerar PDF
    pdf_checkbox = Checkbox(
        value=GENERATE_PDF,
        description="Gerar PDF do relatório?",
        indent=False,
    )
    
    # Widget: Compatibilidade
    compat_checkbox = Checkbox(
        value=APPLY_COMPATIBILITY_FILTER,
        description="Filtrar combos incompatíveis?",
        indent=False,
    )
    
    # Exibir widgets
    display(VBox([
        Label("📌 Escolha as opções abaixo e execute a célula:"),
        HBox([method_dropdown, dataset_dropdown]),
        HBox([preset_dropdown]),
        HBox([quick_check_checkbox]),
        strict_checkbox,
        pdf_checkbox,
        compat_checkbox,
    ]))
    
    # Capturar seleções
    SELECTED_METHOD = method_dropdown.value
    SELECTED_DATASET = dataset_dropdown.value
    SELECTED_PRESET = preset_dropdown.value
    RUN_MODE = "quick_check" if quick_check_checkbox.value else "full"
    STRICT_RESULTS = strict_checkbox.value
    GENERATE_PDF = pdf_checkbox.value
    APPLY_COMPATIBILITY_FILTER = compat_checkbox.value

# ============================================================================
# RESOLUÇÃO DE CAMINHO DO DATASET
# ============================================================================

if SELECTED_DATASET in available_datasets:
    SELECTED_DATASET_ROOT = available_datasets[SELECTED_DATASET]
    print(f"✓ Dataset '{SELECTED_DATASET}' encontrado em: {SELECTED_DATASET_ROOT}")
else:
    # Fallback: tentar adivinhar o caminho
    fallback_paths = {
        "blender_synthetic": "./data/blender_synthetic/nerf_synthetic/lego",
        "d_nerf": "./data/d_nerf/bonsai",
        "mipnerf360": "./data/mipnerf360/bicycle",
        "tanks_and_temples": "./data/tanks_and_temples/Barn",
    }
    SELECTED_DATASET_ROOT = fallback_paths.get(SELECTED_DATASET, "./data/" + SELECTED_DATASET)
    print(f"⚠ Dataset '{SELECTED_DATASET}' não encontrado em ./data. Tentaremos: {SELECTED_DATASET_ROOT}")

# ============================================================================
# RESUMO DE SELEÇÃO
# ============================================================================

print("\n" + "=" * 70)
print("RESUMO DA SELEÇÃO")
print("=" * 70)
print(f"Método:           {SELECTED_METHOD}")
print(f"Dataset:          {SELECTED_DATASET}")
print(f"Caminho:          {SELECTED_DATASET_ROOT}")
print(f"Preset:           {SELECTED_PRESET}")
print(f"Modo:             {RUN_MODE}")
print(f"Modo estrito:     {STRICT_RESULTS}")
print(f"Gerar PDF:        {GENERATE_PDF}")
print(f"Filtro compat:    {APPLY_COMPATIBILITY_FILTER}")
print("=" * 70)

In [ ]:
# Download de datasets via catálogo
# Nota: A célula 7 (Descoberta e Seleção) vai definir qual dataset rodar.
# Esta célula tenta baixar datasets disponíveis. Se o seu dataset escolhido já
# existe em ./data, esta célula será rápida (verá que já existe).
print("=" * 70)
print("Preparando datasets...")
print("=" * 70)
print(f"Dataset selecionado para execução: {SELECTED_DATASET}")
print()

subprocess.run([
    sys.executable, "-m", "nvs_benchmark.cli", "install",
    "--catalog-file", "./configs/install_catalog.json",
    "--only", "datasets",
    "--execute"
], check=True)

print("\n✓ Datasets preparados.")

In [ ]:
# Benchmark (usa seleções da Célula 7: método, dataset, preset)
print("=" * 70)
print(f"Executando benchmark: {SELECTED_METHOD} x {SELECTED_DATASET}")
print("=" * 70)

snapshot_file = f"./artifacts/metrics/{RUN_ID}.json"

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "method-run",
    "--method", SELECTED_METHOD,
    "--dataset", SELECTED_DATASET,
    "--root", SELECTED_DATASET_ROOT,
    "--split", "train",
    "--preset", SELECTED_PRESET,
    "--output-dir", "./artifacts",
    "--log-dir", "./logs",
    "--compute-metrics",
    "--snapshot-file", snapshot_file,
    "--append-snapshot",
]

if STRICT_RESULTS:
    cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

print(f"\nExecutando: {' '.join(cmd)}\n")
subprocess.run(cmd, check=True)

print(f"\n✓ Snapshot gerado em: {snapshot_file}")

In [ ]:
# Gerar relatorio HTML (usa seleções da Célula 7)
print("=" * 70)
print("Gerando relatório HTML...")
print("=" * 70)

report_name = f"{RUN_ID}_{SELECTED_METHOD}_{SELECTED_DATASET}_report"

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "report-generate",
    "--snapshot-file", snapshot_file,
    "--output-dir", "./artifacts/reports",
    "--report-name", report_name,
    "--log-dir", "./logs",
]

if not GENERATE_PDF:
    cmd.append("--no-pdf")

if STRICT_RESULTS:
    cmd.extend(["--strict-snapshot", "--min-methods", "1", "--require-finite-metrics"])

print(f"Comando: {' '.join(cmd)}\n")
subprocess.run(cmd, check=True)

report_html = f"./artifacts/reports/{report_name}.html"
print(f"\n✓ Relatório HTML gerado: {report_html}")

In [ ]:
# Exibir o relatório no notebook
print("=" * 70)
print("Carregando relatório...")
print("=" * 70)

from IPython.display import IFrame, display
import os

if not os.path.exists(report_html):
    print(f"✗ Arquivo não encontrado: {report_html}")
    print("  Verifique se a célula anterior executou sem erros.")
else:
    print(f"✓ Abrindo: {report_html}\n")
    display(IFrame(src=report_html, width=1200, height=700))

In [ ]:
# Compactar e baixar artefatos
print("=" * 70)
print("Preparando download de artefatos...")
print("=" * 70)

import pathlib
colab_files_mod = __import__("google.colab", fromlist=["files"])
files = getattr(colab_files_mod, "files")

zip_path = "/content/nvs_benchmark_artifacts"
print(f"Compactando {REPO_DIR}/artifacts...")
archive_file = shutil.make_archive(zip_path, "zip", REPO_DIR, "artifacts")
print(f"Arquivo gerado: {archive_file}")

if pathlib.Path(archive_file).exists():
    print(f"Tamanho: {pathlib.Path(archive_file).stat().st_size / (1024*1024):.1f} MB")
    print("\n↓ Iniciando download...")
    files.download(archive_file)
else:
    print("✗ Arquivo não encontrado!")

In [ ]:
# Backup opcional dos artefatos no Google Drive
print("=" * 70)
print("Backup de resultados")
print("=" * 70)

if USE_GOOGLE_DRIVE:
    print(f"Backup: SIM → {DRIVE_OUTPUT_DIR}")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    metrics_dst = os.path.join(DRIVE_OUTPUT_DIR, "metrics")
    reports_dst = os.path.join(DRIVE_OUTPUT_DIR, "reports")

    if os.path.exists(metrics_dst):
        shutil.rmtree(metrics_dst)
    if os.path.exists(reports_dst):
        shutil.rmtree(reports_dst)

    print(f"Copiando métricas...")
    shutil.copytree("./artifacts/metrics", metrics_dst)
    print(f"Copiando relatórios...")
    shutil.copytree("./artifacts/reports", reports_dst)
    print(f"✓ Backup concluido em: {DRIVE_OUTPUT_DIR}")
else:
    print("Backup: NÃO")
    print("  Para fazer backup no Drive, mude USE_GOOGLE_DRIVE = True na Célula 2")